# Braitenberg Duck Avoidance

In this assignment you will program a **Braitenberg vehicle** that steers
away from yellow rubber ducks.

## How it works

A Braitenberg vehicle reacts directly to sensor readings — no planning
or world model required.  We split the camera image into a **left half**
and a **right half**, count the yellow pixels in each, and feed those
counts to the wheels:

```
left yellow pixels  ──►  left  wheel speed
right yellow pixels ──►  right wheel speed
```

When the duck is on the **left**, the left wheel speeds up more than the
right → the robot turns **right**, away from the duck.  Same logic
applies to the right side.

## Plan
1. Load and inspect a sample image
2. Learn about the HSV colour space
3. Build a yellow colour mask
4. Compute left/right detection counts
5. Compute motor speeds with the Braitenberg formula
6. Test on all big-duck images

## 1 — Setup

In [ ]:
import sys, os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Make sure we can import the student package
notebook_dir = os.path.abspath('.')
project_root = os.path.abspath(os.path.join(notebook_dir, '..', '..', '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

SAMPLES_DIR = os.path.join(project_root, 'tasks', 'assets', 'samples', 'big-duck')
image_paths = sorted([
    os.path.join(SAMPLES_DIR, f)
    for f in os.listdir(SAMPLES_DIR) if f.endswith('.jpg')
])
print(f'Found {len(image_paths)} sample images')

## 2 — Load a sample image

In [ ]:
def load_rgb(path):
    bgr = cv2.imread(path)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

img = load_rgb(image_paths[14])   # big-duck-15: duck close-up

plt.figure(figsize=(8, 5))
plt.imshow(img)
plt.title('Sample image')
plt.axis('off')
plt.show()
print(f'Image shape: {img.shape}  (height x width x channels)')

## 3 — HSV colour space

RGB mixes colour and brightness together, making colour-based thresholding
sensitive to lighting changes.  **HSV** separates them:

| Channel | Meaning | OpenCV range |
|---------|---------|-------------|
| H (Hue) | Which colour (red, yellow, green …) | 0 – 180 |
| S (Saturation) | How vivid / pure the colour is | 0 – 255 |
| V (Value) | How bright | 0 – 255 |

Yellow rubber ducks are typically:
- **H ≈ 10–35** (orange-yellow portion of the colour wheel)
- **S > 50**  (clearly coloured, not grey)
- **V > 80**  (not too dark)

In [ ]:
img_hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
H, S, V = img_hsv[:,:,0], img_hsv[:,:,1], img_hsv[:,:,2]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img);         axes[0].set_title('Original (RGB)'); axes[0].axis('off')
axes[1].imshow(H, cmap='hsv'); axes[1].set_title('H channel');     axes[1].axis('off')
axes[2].imshow(S, cmap='gray'); axes[2].set_title('S channel');     axes[2].axis('off')
axes[3].imshow(V, cmap='gray'); axes[3].set_title('V channel');     axes[3].axis('off')
plt.tight_layout()
plt.show()

## 4 — Build the yellow mask

We use `cv2.inRange` to keep only pixels whose HSV values fall between
a lower and upper threshold.  **Tune the values below** until the duck
body is white and the background is black.

In [ ]:
# --- TUNE THESE ---
hsv_config = {
    'lower_h':  5,  'lower_s':  50, 'lower_v':  80,
    'upper_h': 35,  'upper_s': 255, 'upper_v': 255,
}
# ------------------

lower = np.array([hsv_config['lower_h'], hsv_config['lower_s'], hsv_config['lower_v']], dtype=np.uint8)
upper = np.array([hsv_config['upper_h'], hsv_config['upper_s'], hsv_config['upper_v']], dtype=np.uint8)

mask = cv2.inRange(img_hsv, lower, upper)

# Overlay: colour detected pixels green
overlay = img.copy()
overlay[mask > 0] = [0, 230, 0]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img);        axes[0].set_title('Original');   axes[0].axis('off')
axes[1].imshow(mask, cmap='gray'); axes[1].set_title('Yellow mask'); axes[1].axis('off')
axes[2].imshow(overlay);    axes[2].set_title('Overlay');    axes[2].axis('off')
plt.tight_layout()
plt.show()
print(f'Yellow pixels detected: {np.sum(mask > 0)}')

## 5 — Left / right detection

Split the mask down the middle and count pixels in each half.

In [ ]:
h, w = mask.shape
mid = w // 2

left_count  = int(np.sum(mask[:, :mid] > 0))
right_count = int(np.sum(mask[:, mid:] > 0))

print(f'Left  half: {left_count} yellow pixels')
print(f'Right half: {right_count} yellow pixels')
print(f'Duck is more on the {"LEFT" if left_count > right_count else "RIGHT"} side')

# Annotate image with left/right split
annotated = img.copy()
annotated[mask > 0] = [0, 230, 0]
annotated[:, mid-1:mid+1] = [255, 0, 0]   # red divider

plt.figure(figsize=(8, 5))
plt.imshow(annotated)
plt.title(f'Left: {left_count} px   |   Right: {right_count} px')
plt.axis('off')
plt.show()

## 6 — Compute motor speeds

The Braitenberg formula:

```
left_speed  = const + gain * (left_pixels  / half_image_pixels)
right_speed = const + gain * (right_pixels / half_image_pixels)
```

- `const` — base forward speed (robot always moves forward a little)
- `gain`  — how aggressively to react

When the duck is on the **left**: `left_speed > right_speed` → turns **right** ✓

In [ ]:
from tasks.braitenberg.packages.braitenberg import get_yellow_mask, get_motor_speeds

bconfig = {'const': 0.3, 'gain': 1.5, 'detection_threshold': 100.0}

left_speed, right_speed = get_motor_speeds(img, hsv_config, bconfig)

print(f'Left  wheel speed: {left_speed:+.3f}')
print(f'Right wheel speed: {right_speed:+.3f}')
if left_speed > right_speed:
    print('→ Robot turns RIGHT (duck detected on left)')
elif right_speed > left_speed:
    print('→ Robot turns LEFT  (duck detected on right)')
else:
    print('→ Robot goes STRAIGHT (no duck or duck centred)')

## 7 — Batch test on all big-duck images

Run the full pipeline on every sample image and display the results.

In [ ]:
n = len(image_paths)
cols = 4
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3))
axes = axes.flatten()

for i, path in enumerate(image_paths):
    frame = load_rgb(path)
    mask  = get_yellow_mask(frame, hsv_config)
    l_spd, r_spd = get_motor_speeds(frame, hsv_config, bconfig)

    overlay = frame.copy()
    overlay[mask > 0] = [0, 230, 0]
    h2, w2 = mask.shape
    overlay[:, w2//2 - 1 : w2//2 + 1] = [255, 0, 0]

    axes[i].imshow(overlay)
    axes[i].set_title(f'{os.path.basename(path)}\nL={l_spd:+.2f}  R={r_spd:+.2f}', fontsize=8)
    axes[i].axis('off')

for j in range(n, len(axes)):
    axes[j].axis('off')

plt.suptitle('Duck detection on all big-duck samples', fontsize=13)
plt.tight_layout()
plt.show()

## Next steps

- If some images are missed, widen the HSV range (lower H/S/V threshold).
- If the floor is being falsely detected, narrow the range or raise S/V.
- Once satisfied with the mask, copy the final `hsv_config` values into
  `config/braitenberg_hsv_config.yaml`.
- In the TTF session, use the web dashboard sliders to fine-tune on real
  ducks under lab lighting.

Run the simulation with:
```bash
python launch.py --sim --task braitenberg
```